In [1]:
pip install astropy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


1. Packages

In [2]:
import astropy
import pandas as pd
import numpy as np

In [3]:
print(astropy.__version__)

8.0.1


2. Get API token

3. Ask LCO for frames
          
4. response.json()
        
5. Get "results"

6. Filter configuration_type == EXPOSE

7. For each EXPOSE frame:
   get full FITS header
   get WCSERR + features

8. Put each frame into a dictionary

9. dictionaries → pandas DataFrame

10. 👀 SEE YOUR DATASET AS A TABLE

In [3]:
API_TOKEN = "cfc495c2c8e9c6226912a5f9b4d96cacad82ae8c"

headers = {
    'Authorization': f'Token {API_TOKEN}'
}

In [4]:
import requests
response = requests.get('https://archive-api.lco.global/frames/')

In [6]:
params = {
    'start': '2026-08-04T00:00:00Z',
    'end': '2026-08-05T23:59:59Z',
    'reduction_level': 91,
    'limit': 1000
}

response = requests.get(
    'https://archive-api.lco.global/frames/',
    headers=headers,
    params=params
)

data = response.json()
frames = data['results']

print(len(frames))

1000


In [7]:
exposed_frames = [
    frame for frame in frames
    if frame.get('configuration_type') == 'EXPOSE'
]

print(len(exposed_frames))

74


In [8]:
rows = []

for frame in exposed_frames:
    frame_id = frame['id']

    response = requests.get(
        f'https://archive-api.lco.global/frames/{frame_id}/headers/',
        headers=headers
    )

    headers_data = response.json()['data']

    # Skip frames without WCSERR
    if 'WCSERR' not in headers_data:
        continue

    row = {
        'id': frame_id,
        'basename': frame.get('basename'),
        'observation_date': frame.get('observation_date'),
        'site_id': frame.get('site_id'),
        'telescope_id': frame.get('telescope_id'),
        'instrument_id': frame.get('instrument_id'),

        'WCSERR': headers_data.get('WCSERR'),
        'SATFRAC': headers_data.get('SATFRAC'),
        'EXPTIME': headers_data.get('EXPTIME'),
        'FILTER': headers_data.get('FILTER'),
        'AIRMASS': headers_data.get('AIRMASS'),
        'L1FWHM': headers_data.get('L1FWHM'),
        'MOONFRAC': headers_data.get('MOONFRAC'),
        'MOONDIST': headers_data.get('MOONDIST'),
        'SKYMAG': headers_data.get('SKYMAG'),
        'WMSHUMID': headers_data.get('WMSHUMID'),
        'WMSTEMP': headers_data.get('WMSTEMP'),
        'PIXSCALE': headers_data.get('PIXSCALE')
    }

    rows.append(row)

df = pd.DataFrame(rows)

df

,id,basename,observation_date,site_id,telescope_id,instrument_id,WCSERR,SATFRAC,EXPTIME,FILTER,AIRMASS,L1FWHM,MOONFRAC,MOONDIST,SKYMAG,WMSHUMID,WMSTEMP,PIXSCALE
0,97015961,cpt0m439-sq39-20260805-0341-e91,2026-08-05T23:46:25.308000Z,cpt,0m4a,sq39,0,0.000005,30.000,V,1.263559,3.935753,0.504216,117.924827,21.328665,59.900002,3.3,0.740
1,97015959,cpt0m439-sq39-20260805-0340-e91,2026-08-05T23:45:48.381000Z,cpt,0m4a,sq39,0,0.000006,30.000,rp,1.262222,3.972790,0.504268,117.918714,21.340587,60.099998,3.1,0.740
2,97015865,cpt0m439-sq39-20260805-0327-e91,2026-08-05T23:37:13.327000Z,cpt,0m4a,sq39,0,0.000004,30.000,V,1.244029,4.245916,0.504867,117.849347,21.474556,55.799999,3.7,0.740
3,97015862,cpt0m439-sq39-20260805-0326-e91,2026-08-05T23:36:36.318000Z,cpt,0m4a,sq39,0,0.000006,30.000,rp,1.242769,4.034275,0.504903,117.843076,21.482293,56.200001,3.6,0.740
4,97015836,cpt0m439-sq39-20260805-0325-e91,2026-08-05T23:31:13.907000Z,cpt,0m4a,sq39,0,0.000004,30.000,V,1.232101,5.274017,0.505295,117.799928,21.561059,54.200001,3.7,0.740
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,97011925,cpt1m012-fa06-20260805-0057-e91,2026-08-05T16:54:31.304000Z,cpt,1m0a,fa06,0,0.000003,10.002,rp,1.017825,2.789302,0.541892,169.127798,19.691512,56.700001,8.3,0.389
70,97013018,cpt1m013-fa01-20260805-0057-e91,2026-08-05T16:54:30.412000Z,cpt,1m0a,fa01,0,0.000000,10.000,rp,1.005689,2.277966,0.541892,155.458862,19.691512,56.700001,8.3,0.389
71,97011817,cpt0m439-sq39-20260805-0080-e91,2026-08-05T16:50:54.372000Z,cpt,0m4a,sq39,0,0.000002,10.000,rp,1.053067,3.055770,0.542260,166.536599,19.091720,55.500000,8.5,0.740
72,97011774,coj1m003-fa19-20260805-0275-e91,2026-08-05T16:47:03.886000Z,coj,1m0a,fa19,0,0.000000,10.001,rp,1.008695,4.686088,0.538461,56.108391,19.933127,72.500000,2.2,0.389


In [3]:
df = pd.read_csv('masterSheet.csv')

In [4]:
df

,id,basename,observation_date,site_id,telescope_id,instrument_id,WCSERR,SATFRAC,EXPTIME,FILTER,AIRMASS,L1FWHM,MOONFRAC,MOONDIST,SKYMAG,WMSHUMID,WMSTEMP,PIXSCALE
0,96967173,cpt0m439-sq39-20260803-0164-e91,2026-08-03T23:50:07.095000Z,cpt,0m4a,sq39,0,0.000002,90.000,V,1.194022,3.878642,0.718337,85.244963,19.102008,29.700001,12.2,0.740
1,96967150,cpt0m439-sq39-20260803-0163-e91,2026-08-03T23:44:16.832000Z,cpt,0m4a,sq39,0,0.000005,90.000,V,1.177753,2.785377,0.718629,85.203361,19.140490,28.700001,12.7,0.740
2,96967073,cpt0m439-sq39-20260803-0135-e91,2026-08-03T23:20:06.986000Z,cpt,0m4a,sq39,0,0.000005,90.000,V,1.120187,2.602835,0.719888,85.024512,19.323324,28.600000,12.6,0.740
3,96967071,cpt0m439-sq39-20260803-0134-e91,2026-08-03T23:14:07.395000Z,cpt,0m4a,sq39,0,0.000007,90.000,V,1.108137,2.505512,0.720206,84.979359,19.373839,28.299999,12.7,0.740
4,96967069,cpt0m439-sq39-20260803-0133-e91,2026-08-03T23:08:06.946000Z,cpt,0m4a,sq39,0,0.000007,90.000,V,1.096850,2.599868,0.720528,84.933820,19.426496,28.400000,12.6,0.740
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1471,91038804,cpt1m013-fa01-20260123-0226-e91,2026-01-23T23:36:11.314000Z,cpt,1m0a,fa01,4,0.000082,120.000,rp,1.245803,2.133429,0.260790,103.360265,22.000000,41.000000,13.7,0.389
1472,91038321,cpt1m013-fa01-20260123-0211-e91,2026-01-23T23:10:18.855000Z,cpt,1m0a,fa01,0,0.000032,120.001,rp,1.799856,2.178887,0.258677,106.100060,22.000000,38.900002,14.3,0.389
1473,91038220,cpt1m013-fa01-20260123-0210-e91,2026-01-23T23:07:04.338000Z,cpt,1m0a,fa01,0,0.000000,90.000,gp,1.568867,3.718121,0.258437,116.475758,22.000000,39.000000,14.2,0.778
1474,91038212,cpt1m013-fa01-20260123-0209-e91,2026-01-23T23:05:21.623000Z,cpt,1m0a,fa01,0,0.000007,90.001,rp,1.566864,3.468024,0.258297,116.496288,22.000000,38.700001,14.2,0.778


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1476 entries, 0 to 1475
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                1476 non-null   int64  
 1   basename          1476 non-null   object 
 2   observation_date  1476 non-null   object 
 3   site_id           1476 non-null   object 
 4   telescope_id      1476 non-null   object 
 5   instrument_id     1476 non-null   object 
 6   WCSERR            1476 non-null   int64  
 7   SATFRAC           1476 non-null   float64
 8   EXPTIME           1476 non-null   float64
 9   FILTER            1476 non-null   object 
 10  AIRMASS           1476 non-null   float64
 11  L1FWHM            1475 non-null   float64
 12  MOONFRAC          1476 non-null   float64
 13  MOONDIST          1476 non-null   float64
 14  SKYMAG            1476 non-null   float64
 15  WMSHUMID          1476 non-null   float64
 16  WMSTEMP           1476 non-null   float64


In [6]:
df.describe()

,id,WCSERR,SATFRAC,EXPTIME,AIRMASS,L1FWHM,MOONFRAC,MOONDIST,SKYMAG,WMSHUMID,WMSTEMP,PIXSCALE
count,1.476000e+03,1476.000000,1.476000e+03,1476.000000,1476.000000,1475.000000,1476.000000,1476.000000,1476.000000,1476.000000,1476.000000,1476.000000
mean,9.694581e+07,0.046070,1.860384e-05,37.872665,1.244001,2.898159,0.209914,96.214821,21.639184,42.132520,13.548374,0.663812
std,8.660041e+05,0.426946,6.220225e-05,54.599272,0.238623,0.923453,0.156184,32.452963,0.655804,23.498737,4.897522,0.163022
min,9.103817e+07,0.000000,0.000000e+00,0.100000,1.000022,0.905180,0.041890,15.230373,18.460374,1.500000,2.100000,0.267000
25%,9.706320e+07,0.000000,3.472222e-07,10.000000,1.041834,2.373571,0.096572,82.472162,21.583882,26.175001,9.700000,0.740000
50%,9.708116e+07,0.000000,3.819444e-06,10.000000,1.185295,2.779660,0.157322,96.475480,22.000000,35.700000,15.400000,0.740000
75%,9.709142e+07,0.000000,6.597222e-06,40.000000,1.375501,3.389833,0.262363,117.188063,22.000000,68.800001,17.500000,0.740000
max,9.710559e+07,4.000000,5.288194e-04,320.000000,2.798195,6.930692,0.744316,178.710843,22.000000,81.300000,23.500000,0.778000


In [7]:
print(df.isna().mean() * 100)

id                  0.000000
basename            0.000000
observation_date    0.000000
site_id             0.000000
telescope_id        0.000000
instrument_id       0.000000
WCSERR              0.000000
SATFRAC             0.000000
EXPTIME             0.000000
FILTER              0.000000
AIRMASS             0.000000
L1FWHM              0.067751
MOONFRAC            0.000000
MOONDIST            0.000000
SKYMAG              0.000000
WMSHUMID            0.000000
WMSTEMP             0.000000
PIXSCALE            0.000000
dtype: float64


In [8]:
print("--- DATASET SHAPE ---")
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}\n")

print("--- MISSING VALUES (%) ---")
missing_pct = df.isna().mean() * 100
print(missing_pct[missing_pct > 0]) # Only shows columns with missing data
print("\n")

print("--- DUPLICATE ROWS ---")
print(f"Total duplicate rows: {df.duplicated().sum()}\n")

print("--- NUMERICAL SUMMARY ---")
display(df.describe())


--- DATASET SHAPE ---
Rows: 1476 | Columns: 18

--- MISSING VALUES (%) ---
L1FWHM    0.067751
dtype: float64


--- DUPLICATE ROWS ---
Total duplicate rows: 24

--- NUMERICAL SUMMARY ---


,id,WCSERR,SATFRAC,EXPTIME,AIRMASS,L1FWHM,MOONFRAC,MOONDIST,SKYMAG,WMSHUMID,WMSTEMP,PIXSCALE
count,1.476000e+03,1476.000000,1.476000e+03,1476.000000,1476.000000,1475.000000,1476.000000,1476.000000,1476.000000,1476.000000,1476.000000,1476.000000
mean,9.694581e+07,0.046070,1.860384e-05,37.872665,1.244001,2.898159,0.209914,96.214821,21.639184,42.132520,13.548374,0.663812
std,8.660041e+05,0.426946,6.220225e-05,54.599272,0.238623,0.923453,0.156184,32.452963,0.655804,23.498737,4.897522,0.163022
min,9.103817e+07,0.000000,0.000000e+00,0.100000,1.000022,0.905180,0.041890,15.230373,18.460374,1.500000,2.100000,0.267000
25%,9.706320e+07,0.000000,3.472222e-07,10.000000,1.041834,2.373571,0.096572,82.472162,21.583882,26.175001,9.700000,0.740000
50%,9.708116e+07,0.000000,3.819444e-06,10.000000,1.185295,2.779660,0.157322,96.475480,22.000000,35.700000,15.400000,0.740000
75%,9.709142e+07,0.000000,6.597222e-06,40.000000,1.375501,3.389833,0.262363,117.188063,22.000000,68.800001,17.500000,0.740000
max,9.710559e+07,4.000000,5.288194e-04,320.000000,2.798195,6.930692,0.744316,178.710843,22.000000,81.300000,23.500000,0.778000


In [ ]:
from google.colab import sheets



sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/1CBiGS4u1b7dNrGmmC9Z58QqDufhQpGv7CRn2dJSONOw/edit#gid=0
